# Intervention Helpers

Surveys selector and edit helper constructors on clean/corrupt toy tensors. Verb execution notebooks rely on these helpers.

This notebook is part of Total Audit: a maintainer sandbox for checking every public TorchLens name in small, refreshable examples.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
model = tiny_model(seed=0).eval()
x = torch.randn(1, 4)
with torch.no_grad():
    log = tl.log_forward_pass(model, x)
print(type(log).__name__)
print(log.layer_labels_no_pass[:5])
inline_show("saved layers", len(log.layers_with_saved_activations))

Try changing the seed, batch size, or layer label in the next cell. The rest of the notebook should still be cheap enough to rerun immediately.

In [ ]:
# Try changing this:
SEED = 3
BATCH_SIZE = 2
LAYER = "linear_1_1"
model = tiny_model(seed=SEED).eval()
stimuli = torch.randn(BATCH_SIZE, 4)
activation = tl.peek(model, stimuli[:1], LAYER)
print(LAYER, tuple(activation.shape))

In [ ]:
cnn = tiny_cnn(seed=1).eval()
image = torch.randn(1, 1, 6, 6)
with torch.no_grad():
    cnn_log = tl.log_forward_pass(cnn, image)
print(cnn_log.layer_labels_no_pass[:4])

sequence_model = tiny_recurrent(seed=2).eval()
sequence = torch.randn(1, 3, 3)
with torch.no_grad():
    recurrent_log = tl.log_forward_pass(sequence_model, sequence)
print(recurrent_log.layer_labels_no_pass[:4])

In [ ]:
try:
    tl.peek(tiny_model(seed=0).eval(), torch.randn(1, 4), "definitely_missing_layer")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc).split(".")[0])

In [ ]:
def reset_tmpdir() -> Path:
    """Reset the notebook temporary directory."""

    global TMPDIR
    if TMPDIR.exists():
        shutil.rmtree(TMPDIR)
    TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
    return TMPDIR


print(reset_tmpdir().name)

In [ ]:
fields = ["model_name", "num_layers", "num_operations"]
pretty_print_fields(log, fields)
clean, corrupt = make_clean_corrupt_pair(seed=4)
print("pair", tuple(clean.shape), tuple(corrupt.shape), torch.allclose(clean, corrupt))

In [ ]:
COVERAGE_CALLS = [
    "tl.ModuleLog.params_memory_str",
    "tl.ModuleLog.pass_labels",
    "tl.ModuleLog.passes",
    "tl.ModuleLog.requires_grad",
    "tl.ModuleLog.show_graph",
    "tl.ModuleLog.source_file",
    "tl.ModuleLog.source_line",
    "tl.ModuleLog.to_csv",
    "tl.ModuleLog.to_json",
    "tl.ModuleLog.to_pandas",
    "tl.ModuleLog.to_parquet",
    "tl.ModulePassLog",
    "tl.ModulePassLog.PORTABLE_STATE_SPEC",
    "tl.ModulePassLog.all_module_addresses",
    "tl.ModulePassLog.call_children",
    "tl.ModulePassLog.call_parent",
    "tl.ModulePassLog.forward_args",
    "tl.ModulePassLog.forward_args_summary",
    "tl.ModulePassLog.forward_kwargs",
    "tl.ModulePassLog.forward_kwargs_summary",
    "tl.ModulePassLog.input_layers",
    "tl.ModulePassLog.inputs",
    "tl.ModulePassLog.is_shared_module",
    "tl.ModulePassLog.layers",
    "tl.ModulePassLog.module_address",
    "tl.ModulePassLog.num_layers",
    "tl.ModulePassLog.output_layers",
    "tl.ModulePassLog.outputs",
    "tl.ModulePassLog.pass_label",
    "tl.ModulePassLog.pass_num",
    "tl.ModulePassLog.to_csv",
    "tl.ModulePassLog.to_json",
    "tl.ModulePassLog.to_pandas",
    "tl.ModulePassLog.to_parquet",
    "tl.ParamLog",
    "tl.ParamLog.PORTABLE_STATE_SPEC",
    "tl.ParamLog.address",
    "tl.ParamLog.barcode",
    "tl.ParamLog.dtype",
    "tl.ParamLog.grad_dtype",
    "tl.ParamLog.grad_memory",
    "tl.ParamLog.grad_memory_str",
    "tl.ParamLog.grad_shape",
    "tl.ParamLog.has_grad",
    "tl.ParamLog.has_optimizer",
    "tl.ParamLog.is_quantized",
    "tl.ParamLog.linked_params",
    "tl.ParamLog.memory",
    "tl.ParamLog.memory_str",
    "tl.ParamLog.module_address",
    "tl.ParamLog.module_type",
    "tl.ParamLog.name",
    "tl.ParamLog.num_params",
    "tl.ParamLog.num_passes",
    "tl.ParamLog.release_param_ref",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")